## Capstone Project: Voice Cloning EDA & Model Comparison (Sample Data + Evaluation Placeholders)
This notebook generates sample voice data for 5 speakers, performs EDA, and provides placeholders for Your-tts, vts inference and evaluation.

In [ ]:
!pip install TTS
!pip install noisereduce
!pip install pandas
!pip install librosa
!pip install seaborn
!pip install speechbrain torchaudio # To analyse voice cloning quality


## Data preview (Audio files and speakers Preview)


In [ ]:
import os
import pandas as pd
from pathlib import Path
import librosa

DATA_DIR = Path('data')
file_paths = []
speakers = []
durations = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".wav"):
            path = os.path.join(root, file)
            y, sr = librosa.load(path, sr=16000)
            file_paths.append(path)
            speakers.append(os.path.basename(root))
            durations.append(len(y)/sr)

df = pd.DataFrame({"path": file_paths, "speaker": speakers, "duration": durations})
print("Dataset preview:")
df.head()
print("\nTotal speakers:", df['speaker'].nunique())
print("Total audio files:", len(df))


## Exploratory Data Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Distribution of audio duration
plt.figure(figsize=(8,5))
sns.histplot(df['duration'], bins=10)
plt.title("Distribution of Audio Durations")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.show()

# Speaker distribution
plt.figure(figsize=(8,4))
sns.countplot(y='speaker', data=df, order=df['speaker'].value_counts().index)
plt.title("Number of Samples per Speaker")
plt.show()


## Feature Extraction & Baseline Speaker Classification

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score


def extract_mfcc(path, n_mfcc=13):
    y, sr = librosa.load(path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfcc.T, axis=0)

X = np.array([extract_mfcc(p) for p in df['path']])
y = df['speaker']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Baseline classifier
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
print("Baseline F1-score:", f1_score(y_test, y_pred, average='weighted'))


## Model Inference Placeholders

In [ ]:
#Testing TTS with simple model for voice cloning

from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/your_tts")
tts.tts_to_file(
    text="This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar. Beyond an event is Knowledge. Beyond an object is Infinity. Beyond a person is Love. Knowledge is beyond events. Every event colours your awareness in some way. ",
    speaker_wav="data/Sample_voice5/p330_002.wav",
    file_path="data/Sample_voice5/results/your_tts_p330_002_out.wav",
    language="en")

## Analyzing voice cloning quality

In [ ]:
#voice cloning for your_tts
import os
import glob
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.pretrained import EncoderClassifier
import logging
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.fetching").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.checkpoints").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)


logging.basicConfig(level=logging.WARNING)

# --------------------------
# Load SpeechBrain ECAPA-TDNN
# --------------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa"
)

# --------------------------
# Settings
# --------------------------
num_voices = 5
pages = ["Page1", "Page2", "Page3"]
model_name = "your_tts"  # update if using multiple models
results_list = []

# --------------------------
# Loop through voices & pages
# --------------------------
for v in range(1, num_voices + 1):
    voice_dir = f"data/Sample_voice{v}"
    results_dir = os.path.join(voice_dir, "results")
    
    # Find original voice ending with _001.wav
    original_files = glob.glob(os.path.join(voice_dir, "*_001.wav"))
    if not original_files:
        print(f" No original voice found in {voice_dir}, skipping...")
        continue
    original_wav = original_files[0]
    
    # Load original audio
    signal, fs = torchaudio.load(original_wav)
    org_emb = spk_model.encode_batch(signal)
    org_emb_1d = org_emb.mean(dim=[0, 2])
    if org_emb.dim() > 1:
        org_emb = org_emb.mean(dim=0)  # reduce to 1D vector
    
    for page in pages:
        cloned_file = os.path.join(
            results_dir,
            f"{os.path.splitext(os.path.basename(original_wav))[0]}_{page}_{model_name}.wav"
        )
        if not os.path.exists(cloned_file):
            print(f" Cloned file not found: {cloned_file}, skipping...")
            continue
        
        # Load cloned audio
        cloned_signal, _ = torchaudio.load(cloned_file)
        cloned_emb = spk_model.encode_batch(cloned_signal)
        cloned_emb_1d = cloned_emb.mean(dim=[0, 2])
        if cloned_emb.dim() > 1:
            cloned_emb = cloned_emb.mean(dim=0)
        
        # Cosine similarity
        similarity = torch.nn.functional.cosine_similarity(
            org_emb_1d.unsqueeze(0),
            cloned_emb_1d.unsqueeze(0),
            dim=1).item()
        
        results_list.append({
            "Voice": f"Sample_voice{v}",
            "Page": page,
            "Original File": original_wav,
            "Cloned File": cloned_file,
            "Cosine Similarity": round(similarity, 4)
        })

your_tts_df = pd.DataFrame(results_list)

## Analytics and comparison between on Voice cloning quality for each voice samples

In [ ]:
%matplotlib inline
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --------------------------
# Quality categorization
# --------------------------
def cloning_quality(sim):
    if sim >= 0.85:
        return "Excellent"
    elif sim >= 0.7:
        return "Moderate"
    else:
        return "Poor"

your_tts_df["Quality"] = your_tts_df["Cosine Similarity"].apply(cloning_quality)
quality_colors = {"Excellent": "green", "Moderate": "orange", "Poor": "red"}

# --------------------------
# Chart 1: Per Page with Quality Dots
# --------------------------
plt.figure(figsize=(12, 5))
sns.barplot(
    data=your_tts_df,
    x="Page",
    y="Cosine Similarity",
    hue="Voice",
    palette="tab10"
)

for i, row in your_tts_df.iterrows():
    plt.scatter(
        row["Page"],
        row["Cosine Similarity"],
        color=quality_colors[row["Quality"]],
        s=100,
        edgecolor="black",
        zorder=5
    )
    plt.text(
        row["Page"],
        row["Cosine Similarity"] + 0.01,
        f"{row['Cosine Similarity']:.2f}",
        ha="center", va="bottom", fontsize=8
    )

plt.title("Voice Cloning Similarity per Page with Quality Indicator")
plt.ylim(0, 1)
plt.ylabel("Cosine Similarity (1 = identical)")
plt.xlabel("Text/Page")
plt.legend(title="Voice")
plt.tight_layout()
plt.savefig("voice_similarity_per_page.png", dpi=300, bbox_inches="tight")
plt.show(block=True)

## Multimodel comparisons with given sample voices

In [ ]:
# Cloned voice wav files are generated under data/sample_voice<number>/results/<file_name>_<model_name>_<page_name>.wav
%matplotlib inline
import os
import glob
import torch
import torchaudio
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from TTS.api import TTS
from speechbrain.pretrained import EncoderClassifier
import logging
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.fetching").setLevel(logging.WARNING)

logging.basicConfig(level=logging.WARNING)

# --------------------------
# Settings
# --------------------------
num_voices = 5
pages = ["Page1", "Page2", "Page3"]
texts = {
    "Page1" : "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar. Beyond an event is Knowledge. Beyond an object is Infinity. Beyond a person is Love. Knowledge is beyond events. Every event colours your awareness in some way. ",
    "Page2" : "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts. Just being on the levels of formal and informal communication cannot make you feel close. 'How are you?' 'Where are you going?' 'How have you been?'",
    "Page3" : "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot. When you prepare a dessert, if sugar or ghee is too little, you can add more. If some other ingredient is too much, it can all be adjusted and repaired. But once it is cooked, it cannot be reversed. Milk can become sweet yoghurt or sour yoghurt, and sour yoghurt can be sweetened." 

}

# Define TTS models
tts_models = [
    "tts_models/multilingual/multi-dataset/your_tts",
    #"tts_models/multilingual/multi-dataset/xtts_v2"
    "tts_models/en/ljspeech/vits"
]

# --------------------------
# Load Speaker Embedding Model
# --------------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa"
)

# --------------------------
# Run TTS and Compute Similarity
# --------------------------
results_list = []

for v in range(1, num_voices + 1):
    voice_dir = f"data/Sample_voice{v}"
    results_dir = os.path.join(voice_dir, "results")
    os.makedirs(results_dir, exist_ok=True)

    # Find original voice file ending with _001.wav
    original_files = glob.glob(os.path.join(voice_dir, "*_001.wav"))
    if not original_files:
        print(f"No original voice found in {voice_dir}, skipping...")
        continue
    original_wav = original_files[0]

    # Load original audio
    signal, fs = torchaudio.load(original_wav)
    org_emb = spk_model.encode_batch(signal)
    if org_emb.dim() > 1:
        org_emb = org_emb.mean(dim=[0, 2])  # 1D vector

    for model_id in tts_models:
        tts_model_name = model_id.split("/")[-1]
        tts = TTS(model_id)

        for page_name in pages:
            # Output file
            output_file = os.path.join(
                results_dir,
                f"{os.path.splitext(os.path.basename(original_wav))[0]}_{page_name}_{tts_model_name}.wav"
            )

            # Generate TTS if file does not exist
            if not os.path.exists(output_file):
                if "multilingual" in model_id:
                    tts.tts_to_file(
                        text=texts[page_name],
                        speaker_wav=original_wav,
                        file_path=output_file,
                        language="en")
                elif "xtts_v2" in model_id:
                    import TTS.tts.configs.xtts_config
                    torch.serialization.safe_globals([TTS.tts.configs.xtts_config.XttsConfig])
                    tts.tts_to_file(
                        text=texts[page_name],
                        speaker_wav=original_wav,
                        file_path=output_file,
                        weights_only=True,
                        language="en")

                else:
                    tts.tts_to_file(
                        text=texts[page_name],
                        speaker_wav=original_wav,
                        file_path=output_file)

            # Load cloned audio
            cloned_signal, _ = torchaudio.load(output_file)
            cloned_emb = spk_model.encode_batch(cloned_signal)
            if cloned_emb.dim() > 1:
                cloned_emb = cloned_emb.mean(dim=[0, 2])

            # Cosine similarity
            similarity = torch.nn.functional.cosine_similarity(
                org_emb.unsqueeze(0),
                cloned_emb.unsqueeze(0),
                dim=1).item()

            results_list.append({
                "Voice": f"Sample_voice{v}",
                "Page": page_name,
                "Model": tts_model_name,
                "Original File": original_wav,
                "Cloned File": output_file,
                "Cosine Similarity": round(similarity, 4)
            })

# --------------------------
# Save results to CSV
# --------------------------
df = pd.DataFrame(results_list)
df.to_csv("multi_model_voice_similarity.csv", index=False)
print(" Multi-model similarity results saved to CSV")
print(df.head())


## Voice Cloning Quality analysis and comparisons - Multimodel

In [ ]:
#Voice Cloning Similarity Heatmap (Models × Pages)
plt.figure(figsize=(10, 6))
pivot_df = df.pivot_table(
    index="Model", columns="Page", values="Cosine Similarity", aggfunc="mean"
)
sns.heatmap(pivot_df, annot=True, cmap="YlGnBu", vmin=0, vmax=1, cbar=True, linewidths=0.5)
plt.title("Voice Cloning Similarity Heatmap (Models × Pages)", fontsize=14)
plt.ylabel("TTS Model")
plt.xlabel("Text/Page")
plt.tight_layout()
plt.show()

In [ ]:
#Voice Cloning Quality Classification per Model
def cloning_quality(sim):
    if sim >= 0.85:
        return "Excellent"
    elif sim >= 0.7:
        return "Moderate"
    else:
        return "Poor"

df["Quality"] = df["Cosine Similarity"].apply(cloning_quality)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df,
    x="Model",
    y="Cosine Similarity",
    hue="Quality",
    style="Page",
    size="Cosine Similarity",
    sizes=(50, 200),
    palette={"Excellent": "green", "Moderate": "orange", "Poor": "red"}
)
plt.title("Voice Cloning Quality Classification per Model", fontsize=14)
plt.ylim(0, 1)
plt.ylabel("Cosine Similarity (1 = identical)")
plt.xlabel("TTS Model")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
#Voice Cloning Quality per Voice and Model
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assume df is already created
# Columns: ["Voice", "Page", "Model", "Cosine Similarity"]

# --------------------------
# Add Quality Labels
# --------------------------
def cloning_quality(sim):
    if sim >= 0.85:
        return "Excellent"
    elif sim >= 0.7:
        return "Moderate"
    else:
        return "Poor"

df["Quality"] = df["Cosine Similarity"].apply(cloning_quality)

# --------------------------
# 1. Grouped Bar Plot per Voice × Model
# --------------------------
plt.figure(figsize=(14, 6))
sns.barplot(
    data=df,
    x="Voice",
    y="Cosine Similarity",
    hue="Model",
    errorbar=None,
    palette="tab10"
)

# Overlay quality dots
quality_colors = {"Excellent": "green", "Moderate": "orange", "Poor": "red"}
for i, row in df.iterrows():
    plt.scatter(
        x=row["Voice"],
        y=row["Cosine Similarity"],
        color=quality_colors[row["Quality"]],
        s=100,
        edgecolor="black",
        zorder=5
    )

plt.title("Voice Cloning Quality per Voice and Model", fontsize=14)
plt.ylim(0, 1)
plt.ylabel("Cosine Similarity (1 = identical)")
plt.xlabel("Reference Voice")
plt.legend(title="TTS Model", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("voice_quality_barplot.png", dpi=300)
plt.show()

# --------------------------
# 2. Heatmap: Voices × Models
# --------------------------
plt.figure(figsize=(10, 6))
pivot_df = df.pivot_table(
    index="Voice", columns="Model", values="Cosine Similarity", aggfunc="mean"
)
sns.heatmap(
    pivot_df, annot=True, cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.5
)

plt.title("Average Voice Cloning Quality (Voices × Models)", fontsize=14)
plt.ylabel("Reference Voice")
plt.xlabel("TTS Model")
plt.tight_layout()
plt.savefig("voice_quality_heatmap.png", dpi=300)
plt.show()


## Final CAPSTONE

In [ ]:
import os, time
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS
import warnings
warnings.filterwarnings("ignore")

# --------------------------
# Setup models
# --------------------------
models = {
    "your_tts": "tts_models/multilingual/multi-dataset/your_tts",
    "vits": "tts_models/en/ljspeech/vits"
}

# Speaker verification model
spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")

# Pages (texts for synthesis)
texts = {
    "Page1": "This is the first test page for benchmarking voice cloning quality.",
    "Page2": "The second page is slightly longer to test runtime performance.",
    "Page3": "Finally, this is page three with another variation in sentence structure."
}

# Data paths
base_dir = "Data"
voices = ["Sample_voicep225","Sample_voicep226","Sample_voicep227","Sample_voicep228","Sample_voicep229","Sample_voicep230",
          "Sample_voicep231","Sample_voicep232","Sample_voicep233","Sample_voicep234","Sample_voicep236","Sample_voicep237",
          "Sample_voicep238","Sample_voicep239","Sample_voicep240","Sample_voicep241","Sample_voicep243","Sample_voicep244",
          "Sample_voicep245","Sample_voicep340"]  # extend for multiple voices

results = []

# --------------------------
# Helper: compute embedding
# --------------------------
def get_embedding(wav_path):
    signal, fs = torchaudio.load(wav_path)
    emb = spk_model.encode_batch(signal).squeeze(0)
    return emb.mean(dim=0) if emb.ndim > 1 else emb

# --------------------------
# Run experiments
# --------------------------
for model_name, model_id in models.items():
    tts = TTS(model_id)

    for voice in voices:
        voice_dir = os.path.join(base_dir, voice)
        ref_files = [f for f in os.listdir(voice_dir) if f.endswith(".wav")]

        # Case 1: Single reference voice
        single_ref = os.path.join(voice_dir, ref_files[0])
        single_emb = get_embedding(single_ref)

        # Case 2: Multiple reference voices (averaged embeddings)
        multi_embs = [get_embedding(os.path.join(voice_dir, f)) for f in ref_files]
        multi_emb = torch.stack(multi_embs).mean(dim=0)

        for case, emb in [("Single", single_emb), ("Multiple", multi_emb)]:
            for page, text in texts.items():
                out_path = os.path.join(
                    voice_dir, "results", f"{os.path.splitext(ref_files[0])[0]}_{model_name}_{page}_{case}.wav"
                )
                os.makedirs(os.path.dirname(out_path), exist_ok=True)

                start = time.time()
                if "multilingual" in model_id:
                    tts.tts_to_file(text=text, speaker_wav=single_ref, file_path=out_path,language = "en")
                else:
                    tts.tts_to_file(text=text, speaker_wav=single_ref, file_path=out_path)
                runtime = time.time() - start

                # Evaluate similarity
                cloned_emb = get_embedding(out_path)
                sim = torch.nn.functional.cosine_similarity(single_emb.unsqueeze(0), cloned_emb.unsqueeze(0)).item()

                results.append({
                    "Model": model_name,
                    "Voice": voice,
                    "Case": case,
                    "Page": page,
                    "Cosine Similarity": round(sim, 4),
                    "Runtime (s)": round(runtime, 2)
                })

# --------------------------
# Save and plot results
# --------------------------
df = pd.DataFrame(results)
df.to_csv("comparison_yourtts_vits.csv", index=False)
print(df)


In [ ]:
%matplotlib inline

print(results)
df.to_csv("comparison_yourtts_vits.csv", index=False)
print(df)

# Plot
sns.catplot(
    data=df, x="Page", y="Cosine Similarity",
    hue="Case", col="Model", kind="bar", height=5, aspect=1
)
plt.show(block=True)

sns.catplot(
    data=df, x="Page", y="Runtime (s)",
    hue="Case", col="Model", kind="bar", height=5, aspect=1
)
plt.show(block=True)

In [ ]:
# Voice Cloning Benchmark Notebook
# Compare Open-source (your_tts, vits) vs Industry APIs (ElevenLabs, OpenAI, Azure)
# Single vs Multiple Sample Voices

import os
import glob
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS

# ==============================
# CONFIG
# ==============================
DATA_DIR = "data"
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

texts = {
    "Page1" : "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar. Beyond an event is Knowledge. Beyond an object is Infinity. Beyond a person is Love. Knowledge is beyond events. Every event colours your awareness in some way. ",
    "Page2" : "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts. Just being on the levels of formal and informal communication cannot make you feel close. 'How are you?' 'Where are you going?' 'How have you been?'",
    "Page3" : "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot. When you prepare a dessert, if sugar or ghee is too little, you can add more. If some other ingredient is too much, it can all be adjusted and repaired. But once it is cooked, it cannot be reversed. Milk can become sweet yoghurt or sour yoghurt, and sour yoghurt can be sweetened."
}

# API KEYS (placeholders, replace with yours)
ELEVENLABS_API_KEY = ""
OPENAI_API_KEY = ""
#AZURE_API_KEY = "your_azure_key"

# Models to benchmark
open_source_models = ["tts_models/en/ljspeech/vits", "tts_models/multilingual/multi-dataset/your_tts"]
industry_models = ["openai_tts", "elevenlabs"]
                   #"azure_tts"]
models =[open_source_models,industry_models]

# ----------------------
# Speaker Embedding Model
# ----------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device":"cpu"}
)


def get_embedding(path):
    signal, fs = torchaudio.load(path)
    emb = spk_model.encode_batch(signal)
    return emb.mean(dim=1).squeeze(0)


# ==============================
# HELPERS
# ==============================

def load_audio(file_path):
    """Load audio and return duration in seconds."""
    y, sr = librosa.load(file_path, sr=None)
    duration = librosa.get_duration(y=y, sr=sr)
    return duration

def benchmark_model(model_id, voices, texts, multi_sample=False):
    """
    Run benchmarking for one model with given voice samples.
    Returns dict with metrics.
    """
    results = []
    start = time.time()
    for voice_dir in voices:
        speaker_files = glob.glob(os.path.join(voice_dir, "*.wav"))

        if not speaker_files:
            continue

        # If multi_sample: average embeddings
        if multi_sample:
            ref_embs = [get_embedding(f) for f in speaker_files]
            org_emb = torch.stack(ref_embs).mean(dim=0)
            ref_wavs = speaker_files
        else:
            org_emb = get_embedding(speaker_files[0])
            ref_wavs = [speaker_files[0]]

        for page, text in texts.items():
            start = time.time()
            for ref_wav in ref_wavs:
                base = os.path.splitext(os.path.basename(ref_wav))[0]
                out_path = os.path.join(
                    voice_dir, "results", f"{base}_{model_id}_{page}.wav"
                )
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
  
    
        # --- Open-source Models ---
        if model_name in open_source_models:
            from TTS.api import TTS
            tts = TTS(model_name)
            out_file = f"{OUTPUT_DIR}/{os.path.basename(sample_files[0])}_{model_name.replace('/','_')}.wav"
            # For simplicity using first file as cloning reference
            #tts.tts_to_file(text=text, speaker_wav=sample_files[0], file_path=out_file)
            try:
                    if "multilingual" in model_id:
                        tts.tts_to_file(text=text, file_path=out_path, speaker_wav=ref_wav, language="en")
                    else:
                        tts.tts_to_file(text=text, file_path=out_path, speaker_wav=ref_wav)

                    cloned_emb = get_embedding(out_path)
                    similarity = torch.nn.functional.cosine_similarity(
                        org_emb.unsqueeze(0), cloned_emb.unsqueeze(0)).item()
            except Exception as e:
                    print(f"Error with {model_id}, {ref_wav}, {page}: {e}")
                    similarity = None
    
        # --- Industry Models (API calls placeholders) ---
        elif model_name == "elevenlabs":
            # Example ElevenLabs request (pseudo, replace with API call)
            time.sleep(2)  # simulate network latency
            out_file = f"{OUTPUT_DIR}/elevenlabs_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        elif model_name == "openai_tts":
            # Example OpenAI request (pseudo, replace with real API call)
            time.sleep(2)
            out_file = f"{OUTPUT_DIR}/openai_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        elif model_name == "azure_tts":
            time.sleep(3)
            out_file = f"{OUTPUT_DIR}/azure_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        end = time.time()
        inference_time = round(end - start, 2)
    
        # Simulated Quality Score (placeholder for MOS or similarity embedding)
        quality_score = np.random.uniform(3, 5) if len(sample_files) > 1 else np.random.uniform(2.5, 4)
    
        # Cost Estimation (hypothetical values)
        if model_name == "elevenlabs":
            cost = 0.20  # per minute of audio
        elif model_name == "openai_tts":
            cost = 0.15
        elif model_name == "azure_tts":
            cost = 0.12
        else:
            cost = 0.0
    
        return {
            "model": model_name,
            "samples_used": len(sample_files),
            "inference_time": inference_time,
            "quality_score": round(quality_score, 2),
            "cost_usd": cost
        }

# ==============================
# RUN BENCHMARK
# ==============================

sample_sets = {
    "single": ["data/sample_voice1/voicefile_001.wav"],
    "multiple": [
        "data/sample_voice1/voicefile_001.wav",
        "data/sample_voice1/voicefile_002.wav",
        "data/sample_voice1/voicefile_003.wav"
    ]
}
results = []

for model in open_source_models + industry_models:
    for setting, files in sample_sets.items():
        res = benchmark_model(model, voice_dirs, texts, multi_sample=False)
        print("RES: " , res)
        #res["setting"] = setting
        results.append(res)
voice_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "sample_voice*")))
# all_results=[]
# for model_name in open_source_models + industry_models:
#     all_results.extend(benchmark_model(model_name, voice_dirs, texts, multi_sample=False))
#     all_results.extend(benchmark_model(model_name, voice_dirs, texts, multi_sample=True))


df = pd.DataFrame(results)
df.to_csv("benchmark_results.csv", index=False)
print("✅ Results saved to benchmark_results.csv")
print(df.head())

# ==============================
# VISUALIZATIONS
# ==============================

# Quality comparison
plt.figure(figsize=(10, 6))
for setting in ["single", "multiple"]:
    subset = df[df["setting"] == setting]
    plt.bar(subset["model"], subset["quality_score"], alpha=0.7, label=f"{setting} samples")
plt.ylabel("Voice Cloning Quality (Simulated MOS)")
plt.title("Quality Comparison: Single vs Multiple Voice Samples")
plt.legend()
plt.xticks(rotation=30)
plt.show()

# Inference time comparison
plt.figure(figsize=(10, 6))
for setting in ["single", "multiple"]:
    subset = df[df["setting"] == setting]
    plt.bar(subset["model"], subset["inference_time"], alpha=0.7, label=f"{setting} samples")
plt.ylabel("Inference Time (s)")
plt.title("Inference Speed Comparison")
plt.legend()
plt.xticks(rotation=30)
plt.show()

# Cost comparison (Industry Models only)
plt.figure(figsize=(10, 6))
subset = df[df["cost_usd"] > 0]
plt.bar(subset["model"], subset["cost_usd"], color="orange")
plt.ylabel("Cost (USD per request)")
plt.title("Industry Model Cost Comparison")
plt.xticks(rotation=30)
plt.show()


In [ ]:
# Voice Cloning Benchmark Notebook
# Compare Open-source (your_tts, vits) vs Industry APIs (ElevenLabs, OpenAI, Azure)
# Single vs Multiple Sample Voices

import os
import glob
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS

# ==============================
# CONFIG
# ==============================
DATA_DIR = "data"
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

texts = {
    "Page1" : "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar. Beyond an event is Knowledge. Beyond an object is Infinity. Beyond a person is Love. Knowledge is beyond events. Every event colours your awareness in some way. ",
    "Page2" : "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts. Just being on the levels of formal and informal communication cannot make you feel close. 'How are you?' 'Where are you going?' 'How have you been?'",
    "Page3" : "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot. When you prepare a dessert, if sugar or ghee is too little, you can add more. If some other ingredient is too much, it can all be adjusted and repaired. But once it is cooked, it cannot be reversed. Milk can become sweet yoghurt or sour yoghurt, and sour yoghurt can be sweetened."
}

# API KEYS (placeholders, replace with yours)
ELEVENLABS_API_KEY = ""
OPENAI_API_KEY = ""
#AZURE_API_KEY = "your_azure_key"

# Models to benchmark
open_source_models = ["tts_models/en/ljspeech/vits", "tts_models/multilingual/multi-dataset/your_tts"]
industry_models = ["openai_tts", "elevenlabs"]
                   #"azure_tts"]
models =[open_source_models,industry_models]

# ----------------------
# Speaker Embedding Model
# ----------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device":"cpu"}
)


def get_embedding(path):
    signal, fs = torchaudio.load(path)
    emb = spk_model.encode_batch(signal)
    return emb.mean(dim=1).squeeze(0)


# ==============================
# HELPERS
# ==============================

def load_audio(file_path):
    """Load audio and return duration in seconds."""
    y, sr = librosa.load(file_path, sr=None)
    duration = librosa.get_duration(y=y, sr=sr)
    return duration

def benchmark_model(model_id, voices, texts, multi_sample=False):
    """
    Run benchmarking for one model with given voice samples.
    Returns dict with metrics.
    """
    results = []
    start = time.time()
    for voice_dir in voices:
        speaker_files = glob.glob(os.path.join(voice_dir, "*.wav"))

        if not speaker_files:
            continue

        # If multi_sample: average embeddings
        if multi_sample:
            ref_embs = [get_embedding(f) for f in speaker_files]
            org_emb = torch.stack(ref_embs).mean(dim=0)
            ref_wavs = speaker_files
        else:
            org_emb = get_embedding(speaker_files[0])
            ref_wavs = [speaker_files[0]]

        for page, text in texts.items():
            start = time.time()
            for ref_wav in ref_wavs:
                base = os.path.splitext(os.path.basename(ref_wav))[0]
                out_path = os.path.join(
                    voice_dir, "results", f"{base}_{model_id}_{page}.wav"
                )
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
  
    
        # --- Open-source Models ---
        if model_name in open_source_models:
            from TTS.api import TTS
            tts = TTS(model_name)
            out_file = f"{OUTPUT_DIR}/{os.path.basename(sample_files[0])}_{model_name.replace('/','_')}.wav"
            # For simplicity using first file as cloning reference
            #tts.tts_to_file(text=text, speaker_wav=sample_files[0], file_path=out_file)
            try:
                    if "multilingual" in model_id:
                        tts.tts_to_file(text=text, file_path=out_path, speaker_wav=ref_wav, language="en")
                    else:
                        tts.tts_to_file(text=text, file_path=out_path, speaker_wav=ref_wav)

                    cloned_emb = get_embedding(out_path)
                    similarity = torch.nn.functional.cosine_similarity(
                        org_emb.unsqueeze(0), cloned_emb.unsqueeze(0)
                    ).item()
            except Exception as e:
                    print(f"Error with {model_id}, {ref_wav}, {page}: {e}")
                    similarity = None
    
        # --- Industry Models (API calls placeholders) ---
        elif model_name == "elevenlabs":
            # Example ElevenLabs request (pseudo, replace with API call)
            time.sleep(2)  # simulate network latency
            out_file = f"{OUTPUT_DIR}/elevenlabs_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        elif model_name == "openai_tts":
            # Example OpenAI request (pseudo, replace with real API call)
            time.sleep(2)
            out_file = f"{OUTPUT_DIR}/openai_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        elif model_name == "azure_tts":
            time.sleep(3)
            out_file = f"{OUTPUT_DIR}/azure_out.wav"
            with open(out_file, "wb") as f:
                f.write(b"FAKE_AUDIO")
    
        end = time.time()
        inference_time = round(end - start, 2)
    
        # Simulated Quality Score (placeholder for MOS or similarity embedding)
        quality_score = np.random.uniform(3, 5) if len(sample_files) > 1 else np.random.uniform(2.5, 4)
    
        # Cost Estimation (hypothetical values)
        if model_name == "elevenlabs":
            cost = 0.20  # per minute of audio
        elif model_name == "openai_tts":
            cost = 0.15
        elif model_name == "azure_tts":
            cost = 0.12
        else:
            cost = 0.0
    
        return {
            "model": model_name,
            "samples_used": len(sample_files),
            "inference_time": inference_time,
            "quality_score": round(quality_score, 2),
            "cost_usd": cost
        }

# ==============================
# RUN BENCHMARK
# ==============================

sample_sets = {
    "single": ["data/sample_voice1/voicefile_001.wav"],
    "multiple": [
        "data/sample_voice1/voicefile_001.wav",
        "data/sample_voice1/voicefile_002.wav",
        "data/sample_voice1/voicefile_003.wav"
    ]
}
results = []

for model in open_source_models + industry_models:
    for setting, files in sample_sets.items():
        res = benchmark_model(model, voice_dirs, texts, multi_sample=False)
        print("RES: " , res)
        #res["setting"] = setting
        results.append(res)
voice_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "sample_voice*")))
# all_results=[]
# for model_name in open_source_models + industry_models:
#     all_results.extend(benchmark_model(model_name, voice_dirs, texts, multi_sample=False))
#     all_results.extend(benchmark_model(model_name, voice_dirs, texts, multi_sample=True))


df = pd.DataFrame(results)
df.to_csv("benchmark_results.csv", index=False)
print("✅ Results saved to benchmark_results.csv")
print(df.head())

# ==============================
# VISUALIZATIONS
# ==============================

# Quality comparison
plt.figure(figsize=(10, 6))
for setting in ["single", "multiple"]:
    subset = df[df["setting"] == setting]
    plt.bar(subset["model"], subset["quality_score"], alpha=0.7, label=f"{setting} samples")
plt.ylabel("Voice Cloning Quality (Simulated MOS)")
plt.title("Quality Comparison: Single vs Multiple Voice Samples")
plt.legend()
plt.xticks(rotation=30)
plt.show()

# Inference time comparison
plt.figure(figsize=(10, 6))
for setting in ["single", "multiple"]:
    subset = df[df["setting"] == setting]
    plt.bar(subset["model"], subset["inference_time"], alpha=0.7, label=f"{setting} samples")
plt.ylabel("Inference Time (s)")
plt.title("Inference Speed Comparison")
plt.legend()
plt.xticks(rotation=30)
plt.show()

# Cost comparison (Industry Models only)
plt.figure(figsize=(10, 6))
subset = df[df["cost_usd"] > 0]
plt.bar(subset["model"], subset["cost_usd"], color="orange")
plt.ylabel("Cost (USD per request)")
plt.title("Industry Model Cost Comparison")
plt.xticks(rotation=30)
plt.show()


In [ ]:
#!pip install openai
!pip install openai

In [ ]:
# ==============================
# Voice Cloning Benchmark Notebook
# Multi-sample per speaker, Open-source + Industry
# ==============================
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS
import logging
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.fetching").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.checkpoints").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)

# ----------------------
# CONFIG
# ----------------------
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

texts = {
    "Page1": "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar...",
    "Page2": "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts...",
    "Page3": "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot..."
}

# ----------------------
# Models
# ----------------------
open_source_models = {
    #"your_tts": "tts_models/multilingual/multi-dataset/your_tts",
    #"vits": "tts_models/en/ljspeech/vits"
}
#industry_models = ["openai_tts", "elevenlabs"]
industry_models = ["elevenlabs"]

# API Keys (replace with yours)
# API keys placeholders
ELEVENLABS_API_KEY = ""
OPENAI_API_KEY = ""


# ----------------------
# Speaker embedding model
# ----------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

def get_embedding(path):
    signal, fs = torchaudio.load(path)
    emb = spk_model.encode_batch(signal)
    return emb.mean(dim=1).squeeze(0)

# ----------------------
# Helper: Industry API TTS
# ----------------------
# OpenAI TTS
import openai
openai.api_key = OPENAI_API_KEY
def generate_openai_tts(text, output_file):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    audio_data = response.read()  # returns bytes
    with open(output_file, "wb") as f:
        f.write(audio_data)

# ElevenLabs TTS
from elevenlabs.client import ElevenLabs
from elevenlabs import save
client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
def generate_elevenlabs_tts(text, voice_id, output_file):
    synthesize_voice(
        text=text,
        voice_id=voice_id,
        #model="eleven_monolingual_v1",
        out_path=output_file)
    with open(output_file, "wb") as f:
        f.write(audio)


def run_elevenlabs_benchmark(sample_voice_paths, texts, output_dir):
    """
    Benchmark ElevenLabs cloning with multiple reference files and generate audio per page.
    """
    client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))

    speaker_id = get_speaker_id(sample_voice_paths)
    # --- Train/IVC clone from reference voices ---
    with open(sample_voice_paths[0], "rb") as f:
        response = client.voices.add(
            name=f"{speaker_id}_voice",
            files=sample_voice_paths  # multiple voice files allowed
        )
    voice_id = response.id  

    results = {}
    for page, text in texts.items():
        start_time = time.time()

        # Synthesize with cloned voice
        audio = client.speech.synthesize(
            voice_id=voice_id,
            model_id="eleven_multilingual_v2",
            text=text
        )

        duration = time.time() - start_time

        out_path = os.path.join(output_dir, f"{speaker_id}_elevenlabs_{page}.wav")
        with open(out_path, "wb") as f:
            f.write(audio)

        results[page] = {
            "file": out_path,
            "time_taken": duration,
            "cost_estimate": len(text) / 1000 * 0.03  # $0.03 per 1k chars approx.
        }
    return results

def create_ivc_voice(name: str, file_paths: list[str], remove_background_noise: bool = False, description: str | None = None):
    """
    Create an instant voice clone (IVC) using ElevenLabs REST API.
    Returns (voice_id, requires_verification)
    Docs: POST https://api.elevenlabs.io/v1/voices/add
    """
    url = "https://api.elevenlabs.io/v1/voices/add"
    # Build multipart list of files: ('files', (filename, fileobj, 'audio/wav'))
    files = []
    fobjs = []
    try:
        for fp in file_paths:
            f = open(fp, "rb")
            fobjs.append(f)
            files.append(("files", (os.path.basename(fp), f, "audio/wav")))
        payload = {
            "name": name,
            "remove_background_noise": str(remove_background_noise).lower()
        }
        if description:
            payload["description"] = description

        resp = requests.post(url, headers=HEADERS, data=payload, files=files, timeout=120)
    finally:
        # close files
        for f in fobjs:
            try:
                f.close()
            except:
                pass

    resp.raise_for_status()
    j = resp.json()
    # Example response: {"voice_id":"...","requires_verification": false}
    return j.get("voice_id")


def synthesize_voice(voice_id: str, text: str, out_path: str, model_id: str = "eleven_multilingual_v2", output_format: str = "mp3_44100_128"):
    """
    Synthesize text using created voice_id.
    Endpoint: POST https://api.elevenlabs.io/v1/text-to-speech/{voice_id}
    Writes audio bytes to out_path (streamed).
    """
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    payload = {
        "text": text,
        "model_id": model_id
    }
    params = {"output_format": output_format}
    headers = {**HEADERS, "Content-Type": "application/json"}
    # stream=True so we can write chunk-by-chunk
    r = requests.post(url, json=payload, headers=headers, params=params, stream=True, timeout=120)
    r.raise_for_status()
    with open(out_path, "wb") as fh:
        for chunk in r.iter_content(chunk_size=4096):
            if chunk:
                fh.write(chunk)
    return out_path

# ==============================
# Benchmark function
# ==============================
def benchmark_model(model_name, voice_dirs, texts):
    print(model_name)
    results = []
    for voice_dir in voice_dirs:
        speaker_files = glob.glob(os.path.join(voice_dir, "*.wav"))
        if not speaker_files:
            continue

        # ----------------------
        # Multi-sample embedding for open-source models
        # ----------------------
        if model_name in open_source_models:
            ref_embs = [get_embedding(f) for f in speaker_files]
            org_emb = torch.stack(ref_embs).mean(dim=0)
            ref_wav = speaker_files[0]  # speaker reference
        else:
            org_emb = None
            ref_wav = None

        for page, text in texts.items():
            base_name = os.path.basename(voice_dir)
            out_dir = os.path.join(voice_dir, "results")
            os.makedirs(out_dir, exist_ok=True)
            out_file = os.path.join(out_dir, f"{base_name}_{model_name}_{page}.wav")

            start_time = time.time()
            similarity = None
            cost = 0.0

            try:
                # ----------------------
                # Open-source TTS
                # ----------------------
                if model_name in open_source_models:
                    tts = TTS(open_source_models[model_name])
                    if "multilingual" in open_source_models[model_name]:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav, language="en")
                    else:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav)
                    # Cosine similarity
                    cloned_emb = get_embedding(out_file)
                    similarity = torch.nn.functional.cosine_similarity(
                        org_emb.unsqueeze(0), cloned_emb.unsqueeze(0)).item()

                # ----------------------
                # OpenAI TTS
                # ----------------------
                elif model_name == "openai_tts":
                    generate_openai_tts(text, out_file)
                    cost = 0.15  # USD per minute

                # ----------------------
                # ElevenLabs TTS
                # ----------------------
                elif model_name == "elevenlabs":
                    print("customVoice_"+base_name)
                    voice_id = create_ivc_voice(
                        name="customVoice_"+base_name,
                        file_paths=speaker_files,  # list of audio samples
                        description="Cloned voice for audiobook generation"
                    )
                    generate_elevenlabs_tts(text, voice_id=voice_id, output_file=out_file)
                    cost = 0.20  # USD per minute

            except Exception as e:
                print(f"Error generating {model_name}, {voice_dir}, {page}: {e}")

            elapsed = round(time.time() - start_time, 2)
            print("Done ", model_name)

            results.append({
                "Model": model_name,
                "Voice": base_name,
                "Page": page,
                "Cosine Similarity": similarity,
                "Time (s)": elapsed,
                "Cost USD": cost
            })

    return results

# ==============================
# Run benchmark
# ==============================
voice_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "Sample_voice*")))
all_results = []

for model_name in list(open_source_models.keys()) + industry_models:
    all_results.extend(benchmark_model(model_name, voice_dirs, texts))

# Save results
df = pd.DataFrame(all_results)
df.to_csv("benchmark_results_multisample.csv", index=False)
print("✅ Results saved to benchmark_results_multisample.csv")
print(df.head())

# ==============================
# Visualizations
# ==============================
sns.set(style="whitegrid")

# 1. Cosine Similarity / Cloning Quality
plt.figure(figsize=(14,6))
sns.barplot(data=df[df["Cosine Similarity"].notnull()], x="Page", y="Cosine Similarity", hue="Model")
plt.title("Voice Cloning Similarity (Cosine) for Open-source Models")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 2. Inference Time
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x="Model", y="Time (s)")
plt.title("Inference Time per Model")
plt.tight_layout()
plt.show()

# 3. Cost comparison (Industry models only)
plt.figure(figsize=(10,6))
subset = df[df["Cost USD"] > 0]
sns.barplot(data=subset, x="Model", y="Cost USD")
plt.title("Industry Model Cost Comparison")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# Voice Cloning Benchmark Notebook
# Multi-sample per speaker, Open-source + Industry
# ==============================
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS
import logging
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.fetching").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.checkpoints").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)

# ----------------------
# CONFIG
# ----------------------
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

texts = {
    "Page1": "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar...",
    "Page2": "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts...",
    "Page3": "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot..."
}

# ----------------------
# Models
# ----------------------
open_source_models = {
    #"your_tts": "tts_models/multilingual/multi-dataset/your_tts",
    #"vits": "tts_models/en/ljspeech/vits"
}
#industry_models = ["openai_tts", "elevenlabs"]
industry_models = ["elevenlabs"]

# API Keys (replace with yours)
# API keys placeholders
ELEVENLABS_API_KEY = ""
OPENAI_API_KEY = ""


# ----------------------
# Speaker embedding model
# ----------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

def get_embedding(path):
    signal, fs = torchaudio.load(path)
    emb = spk_model.encode_batch(signal)
    return emb.mean(dim=1).squeeze(0)

# ----------------------
# Helper: Industry API TTS
# ----------------------
# OpenAI TTS
import openai
openai.api_key = OPENAI_API_KEY
def generate_openai_tts(text, output_file):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    audio_data = response.read()  # returns bytes
    with open(output_file, "wb") as f:
        f.write(audio_data)

# ElevenLabs TTS
from elevenlabs.client import ElevenLabs
from elevenlabs import save
client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
def generate_elevenlabs_tts(text, voice_id, output_file):
    synthesize_voice(
        text=text,
        voice_id=voice_id,
        #model="eleven_monolingual_v1",
        out_path=output_file)
    with open(output_file, "wb") as f:
        f.write(audio)


def run_elevenlabs_benchmark(sample_voice_paths, texts, output_dir):
    """
    Benchmark ElevenLabs cloning with multiple reference files and generate audio per page.
    """
    client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))

    speaker_id = get_speaker_id(sample_voice_paths)
    # --- Train/IVC clone from reference voices ---
    with open(sample_voice_paths[0], "rb") as f:
        response = client.voices.add(
            name=f"{speaker_id}_voice",
            files=sample_voice_paths  # multiple voice files allowed
        )
    voice_id = response.id  

    results = {}
    for page, text in texts.items():
        start_time = time.time()

        # Synthesize with cloned voice
        audio = client.speech.synthesize(
            voice_id=voice_id,
            model_id="eleven_multilingual_v2",
            text=text
        )

        duration = time.time() - start_time

        out_path = os.path.join(output_dir, f"{speaker_id}_elevenlabs_{page}.wav")
        with open(out_path, "wb") as f:
            f.write(audio)

        results[page] = {
            "file": out_path,
            "time_taken": duration,
            "cost_estimate": len(text) / 1000 * 0.03  # $0.03 per 1k chars approx.
        }
    return results

def create_ivc_voice(name: str, file_paths: list[str], remove_background_noise: bool = False, description: str | None = None):
    """
    Create an instant voice clone (IVC) using ElevenLabs REST API.
    Returns (voice_id, requires_verification)
    Docs: POST https://api.elevenlabs.io/v1/voices/add
    """
    url = "https://api.elevenlabs.io/v1/voices/add"
    # Build multipart list of files: ('files', (filename, fileobj, 'audio/wav'))
    files = []
    fobjs = []
    try:
        for fp in file_paths:
            f = open(fp, "rb")
            fobjs.append(f)
            files.append(("files", (os.path.basename(fp), f, "audio/wav")))
        payload = {
            "name": name,
            "remove_background_noise": str(remove_background_noise).lower()
        }
        if description:
            payload["description"] = description

        resp = requests.post(url, headers=HEADERS, data=payload, files=files, timeout=120)
    finally:
        # close files
        for f in fobjs:
            try:
                f.close()
            except:
                pass

    resp.raise_for_status()
    j = resp.json()
    # Example response: {"voice_id":"...","requires_verification": false}
    return j.get("voice_id")


def synthesize_voice(voice_id: str, text: str, out_path: str, model_id: str = "eleven_multilingual_v2", output_format: str = "mp3_44100_128"):
    """
    Synthesize text using created voice_id.
    Endpoint: POST https://api.elevenlabs.io/v1/text-to-speech/{voice_id}
    Writes audio bytes to out_path (streamed).
    """
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    payload = {
        "text": text,
        "model_id": model_id
    }
    params = {"output_format": output_format}
    headers = {**HEADERS, "Content-Type": "application/json"}
    # stream=True so we can write chunk-by-chunk
    r = requests.post(url, json=payload, headers=headers, params=params, stream=True, timeout=120)
    r.raise_for_status()
    with open(out_path, "wb") as fh:
        for chunk in r.iter_content(chunk_size=4096):
            if chunk:
                fh.write(chunk)
    return out_path

# ==============================
# Benchmark function
# ==============================
def benchmark_model(model_name, voice_dirs, texts):
    print(model_name)
    results = []
    for voice_dir in voice_dirs:
        speaker_files = glob.glob(os.path.join(voice_dir, "*.wav"))
        if not speaker_files:
            continue

        # ----------------------
        # Multi-sample embedding for open-source models
        # ----------------------
        if model_name in open_source_models:
            ref_embs = [get_embedding(f) for f in speaker_files]
            org_emb = torch.stack(ref_embs).mean(dim=0)
            ref_wav = speaker_files[0]  # speaker reference
        else:
            org_emb = None
            ref_wav = None

        for page, text in texts.items():
            base_name = os.path.basename(voice_dir)
            out_dir = os.path.join(voice_dir, "results")
            os.makedirs(out_dir, exist_ok=True)
            out_file = os.path.join(out_dir, f"{base_name}_{model_name}_{page}.wav")

            start_time = time.time()
            similarity = None
            cost = 0.0

            try:
                # ----------------------
                # Open-source TTS
                # ----------------------
                if model_name in open_source_models:
                    tts = TTS(open_source_models[model_name])
                    if "multilingual" in open_source_models[model_name]:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav, language="en")
                    else:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav)
                    # Cosine similarity
                    cloned_emb = get_embedding(out_file)
                    similarity = torch.nn.functional.cosine_similarity(
                        org_emb.unsqueeze(0), cloned_emb.unsqueeze(0)).item()

                # ----------------------
                # OpenAI TTS
                # ----------------------
                elif model_name == "openai_tts":
                    generate_openai_tts(text, out_file)
                    cost = 0.15  # USD per minute

                # ----------------------
                # ElevenLabs TTS
                # ----------------------
                elif model_name == "elevenlabs":
                    print("customVoice_"+base_name)
                    voice_id = create_ivc_voice(
                        name="customVoice_"+base_name,
                        file_paths=speaker_files,  # list of audio samples
                        description="Cloned voice for audiobook generation"
                    )
                    generate_elevenlabs_tts(text, voice_id=voice_id, output_file=out_file)
                    cost = 0.20  # USD per minute

            except Exception as e:
                print(f"Error generating {model_name}, {voice_dir}, {page}: {e}")

            elapsed = round(time.time() - start_time, 2)
            print("Done ", model_name)

            results.append({
                "Model": model_name,
                "Voice": base_name,
                "Page": page,
                "Cosine Similarity": similarity,
                "Time (s)": elapsed,
                "Cost USD": cost
            })

    return results

# ==============================
# Run benchmark
# ==============================
voice_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "Sample_voice*")))
all_results = []

for model_name in list(open_source_models.keys()) + industry_models:
    all_results.extend(benchmark_model(model_name, voice_dirs, texts))

# Save results
df = pd.DataFrame(all_results)
df.to_csv("benchmark_results_multisample.csv", index=False)
print("✅ Results saved to benchmark_results_multisample.csv")
print(df.head())

# ==============================
# Visualizations
# ==============================
sns.set(style="whitegrid")

# 1. Cosine Similarity / Cloning Quality
plt.figure(figsize=(14,6))
sns.barplot(data=df[df["Cosine Similarity"].notnull()], x="Page", y="Cosine Similarity", hue="Model")
plt.title("Voice Cloning Similarity (Cosine) for Open-source Models")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 2. Inference Time
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x="Model", y="Time (s)")
plt.title("Inference Time per Model")
plt.tight_layout()
plt.show()

# 3. Cost comparison (Industry models only)
plt.figure(figsize=(10,6))
subset = df[df["Cost USD"] > 0]
sns.barplot(data=subset, x="Model", y="Cost USD")
plt.title("Industry Model Cost Comparison")
plt.tight_layout()
plt.show()


In [ ]:
import os
import glob
import requests
import time

os.environ["ELEVENLABS_API_KEY"]= ""
ELEVEN_API_KEY = os.getenv("ELEVENLABS_API_KEY")  # set this in your environment
if not ELEVEN_API_KEY:
    raise RuntimeError("Set ELEVENLABS_API_KEY env var")

# Example texts (your Page1/2/3)
texts = {
    "Page1": "This Begins a New Practice ...",
    "Page2": "Close to the Master ...",
    "Page3": "Prarabdha Karma and Sanchita Karma ..."
}

HEADERS = {"xi-api-key": ""}

def create_ivc_voice(name: str, file_paths: list[str], remove_background_noise: bool = False, description: str | None = None):
    """
    Create an instant voice clone (IVC) using ElevenLabs REST API.
    Returns (voice_id, requires_verification)
    Docs: POST https://api.elevenlabs.io/v1/voices/add
    """
    url = "https://api.elevenlabs.io/v1/voices/add"
    # Build multipart list of files: ('files', (filename, fileobj, 'audio/wav'))
    files = []
    fobjs = []
    try:
        for fp in file_paths:
            f = open(fp, "rb")
            fobjs.append(f)
            files.append(("files", (os.path.basename(fp), f, "audio/wav")))
        payload = {
            "name": name,
            "remove_background_noise": str(remove_background_noise).lower()
        }
        if description:
            payload["description"] = description

        resp = requests.post(url, headers=HEADERS, data=payload, files=files, timeout=120)
    finally:
        # close files
        for f in fobjs:
            try:
                f.close()
            except:
                pass

    resp.raise_for_status()
    j = resp.json()
    # Example response: {"voice_id":"...","requires_verification": false}
    return j.get("voice_id"), j.get("requires_verification", False)


def synthesize_voice(voice_id: str, text: str, out_path: str, model_id: str = "eleven_multilingual_v2", output_format: str = "mp3_44100_128"):
    """
    Synthesize text using created voice_id.
    Endpoint: POST https://api.elevenlabs.io/v1/text-to-speech/{voice_id}
    Writes audio bytes to out_path (streamed).
    """
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    payload = {
        "text": text,
        "model_id": model_id
    }
    params = {"output_format": output_format}
    headers = {**HEADERS, "Content-Type": "application/json"}
    # stream=True so we can write chunk-by-chunk
    r = requests.post(url, json=payload, headers=headers, params=params, stream=True, timeout=120)
    r.raise_for_status()
    with open(out_path, "wb") as fh:
        for chunk in r.iter_content(chunk_size=4096):
            if chunk:
                fh.write(chunk)
    return out_path


# -------------------------------------------------------
# Example: iterate speaker folders, build IVC from all .wav, then synth per page
# -------------------------------------------------------
base = "data"  # your data folder
voice_dirs = sorted(glob.glob(os.path.join(base, "Sample_voice*")))

for vdir in voice_dirs:
    wavs = sorted(glob.glob(os.path.join(vdir, "*.wav")))
    if not wavs:
        print("no wavs in", vdir); continue

    speaker_name = os.path.basename(vdir)
    print(f"\nCreating IVC for {speaker_name} using {len(wavs)} sample files...")

    try:
        voice_id, needs_verification = create_ivc_voice(name=speaker_name, file_paths=wavs, remove_background_noise=False, description="Capstone voice clone")
    except requests.HTTPError as e:
        print("Create voice failed:", e, e.response.text if hasattr(e, "response") else "")
        continue

    print("voice_id:", voice_id, "requires_verification:", needs_verification)
    if needs_verification:
        print("⚠️ This voice requires manual verification in ElevenLabs dashboard before it can be used. Skipping synthesis for this voice.")
        continue

    # Now synthesize Page1/Page2/Page3 using this voice_id
    results_dir = os.path.join(vdir, "results")
    os.makedirs(results_dir, exist_ok=True)
    for page_name, page_text in texts.items():
        out_name = f"{speaker_name}_elevenlabs_{voice_id}_{page_name}.mp3"  # mp3 default; choose extension accordingly
        out_path = os.path.join(results_dir, out_name)
        try:
            synthesize_voice(voice_id, page_text, out_path)
            print("Saved:", out_path)
        except Exception as e:
            print("Synthesis failed:", e)


In [ ]:
df.to_csv("benchmark_results_full.csv", index=False)
print("✅ Results saved to benchmark_results_full.csv")
print(df.head())

# ==============================
# Visualizations
# ==============================
sns.set(style="whitegrid")

# Voice Cloning Quality
plt.figure(figsize=(14,6))
sns.barplot(data=df, x="Page", y="Cosine Similarity", hue="Model")
plt.title("Voice Cloning Similarity: Open-source vs Industry Models")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Inference Time
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x="Model", y="Time (s)", hue="Mode")
plt.title("Inference Time: Single vs Multi-sample")
plt.tight_layout()
plt.show()

# Cost comparison (industry models)
plt.figure(figsize=(10,6))
subset = df[df["Cost USD"] > 0]
sns.barplot(data=subset, x="Model", y="Cost USD", hue="Mode")
plt.title("Industry Model Cost Comparison")
plt.tight_layout()
plt.show()

In [ ]:
import requests

# Create IVC voice (POST /v1/voices/add)
response = requests.post(
  "https://api.elevenlabs.io/v1/voices/add",
  headers={
    "xi-api-key": ""
  },
  data={
    'name': "Custom_voice",
  },
  files={
    'files': ('p226_001.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_001.wav', 'rb')),
    'files': ('p226_002.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_002.wav', 'rb')),
    'files': ('p226_003.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_003.wav', 'rb')),
    'files': ('p226_004.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_004.wav', 'rb')),
    'files': ('p226_005.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_005.wav', 'rb')),
    'files': ('p226_007.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_006.wav', 'rb')),
    'files': ('p226_006.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_007.wav', 'rb')),
    'files': ('p226_008.wav', open('/Users/Raji/Documents/Raji/Berkely-AIML/Module 24/voice-cloning-TTS-models-final/data/Sample_voicep226/p226_008.wav', 'rb'))
  },
)

print(response.json().get("voice_id"))

In [ ]:
%matplotlib inline
# ==============================
# Voice Cloning Benchmark Notebook
# Multi-sample per speaker, Open-source + Industry
# ==============================

import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.pretrained import EncoderClassifier
from TTS.api import TTS
import requests
import logging

# Suppress warnings from speechbrain
logging.getLogger("speechbrain.utils.parameter_transfer").setLevel(logging.WARNING)
logging.getLogger("speechbrain.dataio.encoder").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.fetching").setLevel(logging.WARNING)
logging.getLogger("speechbrain.utils.checkpoints").setLevel(logging.WARNING)

# ----------------------
# CONFIG
# ----------------------
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

texts = {
    "Page1": "This Begins a New Practice This begins a series of weekly notes of Knowledge from Gurudev Sri Sri Ravi Shankar...",
    "Page2": "Close to the Master If you’re not feeling close to the Master, it’s because of you – because of your mind, your ego concepts...",
    "Page3": "Prarabdha Karma and Sanchita Karma Some karma can be changed and some cannot..."
}

open_source_models = {
    "your_tts": "tts_models/multilingual/multi-dataset/your_tts",
    "vits": "tts_models/en/ljspeech/vits"
}
#eleven_labs_voiceid("p226":"eUMZHdC9ZY4DmIAFXUkk",

#industry_models = ["openai_tts", "elevenlabs"]
industry_models = ["openai_tts"]

# ----------------------
# API Keys (set your own!)
# ----------------------
#ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

HEADERS = {"xi-api-key": ELEVENLABS_API_KEY}

# ----------------------
# Speaker Embeddings
# ----------------------
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

def get_embedding(path):
    signal, fs = torchaudio.load(path)
    emb = spk_model.encode_batch(signal)
    return emb.mean(dim=1).squeeze(0)

# ----------------------
# OpenAI TTS
# ----------------------
import openai
openai.api_key = OPENAI_API_KEY

def generate_openai_tts(text, output_file):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    audio_data = response.read()  # bytes
    with open(output_file, "wb") as f:
        f.write(audio_data)

# ----------------------
# ElevenLabs REST API
# ----------------------
def create_ivc_voice(name: str, file_paths: list[str], description: str = None):
    """
    Create an instant voice clone (IVC) using ElevenLabs REST API.
    """
    url = "https://api.elevenlabs.io/v1/voices/add"
    files = []
    fobjs = []
    try:
        for fp in file_paths:
            f = open(fp, "rb")
            fobjs.append(f)
            files.append(("files", (os.path.basename(fp), f, "audio/wav")))
        payload = {"name": name}
        if description:
            payload["description"] = description
        resp = requests.post(url, headers=HEADERS, data=payload, files=files, timeout=120)
    finally:
        for f in fobjs:
            f.close()
    resp.raise_for_status()
    return resp.json().get("voice_id")

def synthesize_voice(voice_id: str, text: str, out_path: str,
                     model_id: str = "eleven_multilingual_v2",
                     output_format: str = "mp3_44100_128"):
    """
    Synthesize text using created voice_id.
    """
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    payload = {"text": text, "model_id": model_id}
    headers = {**HEADERS, "Content-Type": "application/json"}
    r = requests.post(url, json=payload, headers=headers, stream=True, timeout=120)
    r.raise_for_status()
    with open(out_path, "wb") as fh:
        for chunk in r.iter_content(chunk_size=4096):
            if chunk:
                fh.write(chunk)
    return out_path

# ----------------------
# Benchmark function
# ----------------------
def benchmark_model(model_name, voice_dirs, texts):
    results = []
    for voice_dir in voice_dirs:
        speaker_files = glob.glob(os.path.join(voice_dir, "*.wav"))
        if not speaker_files:
            continue

        base_name = os.path.basename(voice_dir)
        out_dir = os.path.join(voice_dir, "results")
        os.makedirs(out_dir, exist_ok=True)

        # For open-source: multi-sample embedding
        if model_name in open_source_models:
            ref_embs = [get_embedding(f) for f in speaker_files]
            org_emb = torch.stack(ref_embs).mean(dim=0)
            ref_wav = speaker_files[0]
        else:
            org_emb = None
            ref_wav = None

        for page, text in texts.items():
            out_file = os.path.join(out_dir, f"{base_name}_{model_name}_{page}.wav")
            start_time = time.time()
            similarity = None
            cost = 0.0

            try:
                # ----------------------
                # Open-source TTS
                # ----------------------
                if model_name in open_source_models:
                    tts = TTS(open_source_models[model_name])
                    if "multilingual" in open_source_models[model_name]:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav, language="en")
                    else:
                        tts.tts_to_file(text=text, file_path=out_file, speaker_wav=ref_wav)

                    cloned_emb = get_embedding(out_file)
                    similarity = torch.nn.functional.cosine_similarity(
                        org_emb.unsqueeze(0), cloned_emb.unsqueeze(0)).item()

                # ----------------------
                # OpenAI TTS
                # ----------------------
                elif model_name == "openai_tts":
                    generate_openai_tts(text, out_file)
                    cost = 0.15  # USD/min (approx)

                # ----------------------
                # ElevenLabs TTS
                # ----------------------
                elif model_name == "elevenlabs":
                    print(speaker_files[0])
                    voice_id = create_ivc_voice(name=f"customVoice_{base_name}",file_paths=speaker_files[0],description="Cloned voice for audiobook generation")
                    #synthesize_voice(voice_id, text, out_file)
                    cost = len(text) / 1000 * 0.03  # $0.03 per 1k chars

            except Exception as e:
                print(f"❌ Error with {model_name}, {voice_dir}, {page}: {e}")

            elapsed = round(time.time() - start_time, 2)
            results.append({
                "Model": model_name,
                "Voice": base_name,
                "Page": page,
                "Cosine Similarity": similarity,
                "Time (s)": elapsed,
                "Cost USD": cost
            })
    return results

# ----------------------
# Run benchmark
# ----------------------
voice_dirs = sorted(glob.glob(os.path.join(DATA_DIR, "Sample_voice*")))
all_results = []

#for model_name in list(open_source_models.keys()) + industry_models:
#    all_results.extend(benchmark_model(model_name, voice_dirs, texts))

for model_name in list(open_source_models.keys()) + industry_models:
    all_results.extend(benchmark_model(model_name, voice_dirs, texts))

# Save results
df = pd.DataFrame(all_results)
df.to_csv("benchmark_results_multisample.csv", index=False)
print("✅ Results saved to benchmark_results_multisample.csv")
print(df.head())

# ----------------------
# Visualizations
# ----------------------
sns.set(style="whitegrid")

# 1. Cloning Quality (only open-source)
plt.figure(figsize=(14,6))
sns.barplot(data=df[df["Cosine Similarity"].notnull()],
            x="Page", y="Cosine Similarity", hue="Model")
plt.title("Voice Cloning Similarity (Cosine) - Open-source")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 2. Inference Time
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x="Model", y="Time (s)")
plt.title("Inference Time per Model")
plt.tight_layout()
plt.show()

# 3. Cost (Industry models)
plt.figure(figsize=(10,6))
subset = df[df["Cost USD"] > 0]
sns.barplot(data=subset, x="Model", y="Cost USD")
plt.title("Industry Model Cost Comparison")
plt.tight_layout()
plt.show(block=True)
